# Chat Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> FINAL
```


### a) Setup: Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"]="WARNING"

### b) arrange


In [1]:
import os
from gai.asm.agents import ChatAgent
from gai.messages import FileMonologue
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp

# Configure Monologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="Sara", file_path=file_path)

# Configure LLM

llm_config = config_helper.get_client_config(
    {
        "client_type": "anthropic",
        "model": "claude-sonnet-4-20250514",
        "extra": {
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
            "stream": True,
        },
    }
)

agent = ChatAgent(
    agent_name="Sara",
    llm_config=llm_config,
    monologue=monologue
)

# INIT -> IS_TOOL_CALL
resp = await agent.start_async()
assert agent.fsm.state == "IS_TOOL_CALL"
print(f"\ncompleted state: {agent.fsm.state}")
print(f"\nis_tool_call_result: {agent.fsm.state_bag['is_tool_call_result']}")
print(f"\npredicate_result: {agent.fsm.state_bag['predicate_result']}")



completed state: IS_TOOL_CALL

is_tool_call_result: False

predicate_result: False


### b) run and infer the context from dialogue

Create an artificial dialogue about horror stories to show that the context is passed to the agent via the dialogue.

We also initiate a new agent object here to demonstrate stateless agent.



In [2]:
agent = ChatAgent(
    agent_name="Sara",
    llm_config=llm_config,
    monologue=monologue
)

from gai.lib.constants import DEFAULT_GUID
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9',
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'I love horror stories, are you familiar with them?',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37',
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 1,
            'chunk_no': 10,
            'chunk': '<eom>',
            'role': "assistant",
            'content': 'Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?'
        }
    })]

# Create an artificial dialogue history for testing
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages, file_path=file_path)
recap = dialogue.extract_recap()
user_message = "Tell me a one paragraph story."

# IS_TOOL_CALL -> CHAT
resp = await agent.resume_async(user_message=user_message, recap=recap)
async for chunk in resp:
    print(chunk, end="", flush=True)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "CHAT"

# CHAT -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print(chunk, end="", flush=True)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"


assistant_message = agent.fsm.state_bag["get_assistant_message"]()
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(sender="Sara", chunk="<eom>", content=assistant_message)

Here's a horror story for you:

The old music box had been silent for decades, gathering dust in Margaret's attic until the power outage forced her to search for candles. As her trembling fingers lifted the delicate lid, the tiny ballerina inside began to spin to a haunting melody she'd never heard before—impossible, since the spring had been broken for years. The porcelain dancer's painted smile seemed to widen in the flickering candlelight, and Margaret realized with growing terror that the figure was turning not to the rhythm of the music, but to match the frantic beating of her own heart. When she tried to close the lid, it wouldn't budge, and the ballerina's head slowly rotated to face her directly, its glassy eyes reflecting something that definitely wasn't Margaret's own terrified expression. The music grew louder, more discordant, and as Margaret backed away in horror, she heard the unmistakable sound of tiny porcelain feet stepping down from their velvet stage, beginning to da

MessagePydantic(id='325bc1fc-19a9-4686-97da-c31077ddbfb2', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753773797.270501, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.15', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="Here's a horror story for you:\n\nThe old music box had been silent for decades, gathering dust in Margaret's attic until the power outage forced her to search for candles. As her trembling fingers lifted the delicate lid, the tiny ballerina inside began to spin to a haunting melody she'd never heard before—impossible, since the spring had been broken for years. The porcelain dancer's painted smile seemed to widen in the flickering candlelight, and Margaret realized with growing terror that the figure was turning not to the rhythm of the music, but to match the frantic beating of her

### c) Show monologue

In [3]:
import json
from gai.messages import message_helper

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.monologue.list_chat_messages()
for message in messages[-2:]:
    print(json.dumps(message, indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = message_helper.get_messages_length(messages)
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "role": "user",
    "content": "\n            Your name is Sara within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            \n            Tell me a one paragraph story.\n\n            Here is a recap of the conversation:\n            User: I love horror stories, are you familiar with them?\nSara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?\n            \n            \n            You may ask me for more information if you need to clarify my request but ask just enough to get the information you need to get started.\

### d) Show dialogue

In [4]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: I love horror stories, are you familiar with them?
Sara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?
User: Sara, Tell me a one paragraph story.
Sara: Here's a horror story for you:

The old music box had been silent for decades, gathering dust in Margaret's attic until the power outage forced her to search for candles. As her trembling fingers lifted the delicate lid, the tiny ballerina inside began to spin to a haunting melody she'd never heard before—impossible, since the spring had been broken for years. The porcelain dancer's painted smile seemed to widen in the flickering candlelight, and Margaret realized with growing terror that the figure was turning not to the rhythm of the music, but to match the frantic beating of her own heart. When she tried to close the lid, it wouldn't budge, and the balleri